# Análisis Completo del Modelo LSTM para Predicción de MERVAL

Este notebook combina:
- Explicación e interpretación del modelo
- Comparación con y sin features de sentimiento
- Visualizaciones de resultados
- Análisis de hiperparámetros
- Interpretación de probabilidades y predicciones

## Diferencia entre Probabilidad y Accuracy

- **Probabilidad**: Confianza del modelo (0-1). Ej: 0.75 = 75% de confianza de que MERVAL subirá
- **Accuracy**: % de predicciones correctas. Ej: 0.65 = 65% de aciertos

La probabilidad se calcula aplicando sigmoid al logit del modelo. Si probabilidad ≥ 0.5, predice "subirá", si no "bajará".


## 1. Configuración e Importaciones


In [7]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Agregar src al path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Importar módulos del proyecto
from src.LSTM.train_boolean_lstm import (
    TrainingConfig,
    BooleanLSTM,
    load_dataset,
    build_sequences,
    scale_windows,
    set_global_seed,
    evaluate_model,
    train_one_fold
)
import torch
from torch import nn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("✅ Módulos importados correctamente")


✅ Módulos importados correctamente


## 2. Explicación del Modelo

### ¿Qué hace el modelo?

El modelo es una **LSTM (Long Short-Term Memory)** que predice si el índice MERVAL **subirá o bajará** al día siguiente (clasificación binaria).

### Arquitectura:
```
Input: Secuencia de N días × M features
    ↓
LSTM Layer (captura patrones temporales)
    ↓
Dropout (regularización)
    ↓
Fully Connected Layer
    ↓
Output: Logit → Sigmoid → Probabilidad (0-1)
```

### Interpretación:
- Si probabilidad ≥ 0.5 → Predice "subirá" (clase 1)
- Si probabilidad < 0.5 → Predice "bajará" (clase 0)
- Accuracy mide qué tan bien predice en general


## 3. Funciones Auxiliares para Entrenamiento con Probabilidades


In [8]:
def train_model_with_predictions(
    features: np.ndarray,
    labels: np.ndarray,
    config: TrainingConfig,
    return_predictions: bool = True
) -> dict:
    """
    Entrena modelo y retorna métricas + predicciones + probabilidades.
    
    Returns:
        Dict con métricas, predicciones, probabilidades, y_true, y pérdidas
    """
    device = torch.device(config.device)
    
    # Crear secuencias
    sequences, targets = build_sequences(features, labels, config.lookback)
    
    # Split simple (80% train, 20% test)
    split_idx = int(len(sequences) * 0.8)
    X_train_raw = sequences[:split_idx]
    y_train = targets[:split_idx]
    X_test_raw = sequences[split_idx:]
    y_test = targets[split_idx:]
    
    # Escalar
    X_train, X_test, scaler = scale_windows(X_train_raw, X_test_raw)
    
    # Crear y entrenar modelo
    model = BooleanLSTM(
        input_dim=X_train.shape[-1],
        hidden_dim=config.hidden_size,
        dropout=config.dropout
    )
    model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    # Entrenar
    tensor_x = torch.from_numpy(X_train.astype(np.float32))
    tensor_y = torch.from_numpy(y_train.astype(np.float32))
    
    train_losses = []
    for epoch in range(1, config.epochs + 1):
        model.train()
        epoch_losses = []
        
        for i in range(0, len(tensor_x), config.batch_size):
            batch_x = tensor_x[i:i+config.batch_size].to(device)
            batch_y = tensor_y[i:i+config.batch_size].to(device)
            
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            if config.clip_value > 0:
                nn.utils.clip_grad_norm_(model.parameters(), config.clip_value)
            optimizer.step()
            epoch_losses.append(loss.item())
        
        avg_loss = np.mean(epoch_losses)
        train_losses.append(avg_loss)
        if epoch % 5 == 0:
            print(f"Epoch {epoch}/{config.epochs} - Loss: {avg_loss:.4f}")
    
    # Evaluar con probabilidades
    eval_result = evaluate_model(model, X_test, y_test, device, return_predictions=True)
    
    result = {
        'metrics': {
            'accuracy': eval_result['accuracy'],
            'precision': eval_result['precision'],
            'recall': eval_result['recall'],
            'f1': eval_result['f1'],
        },
        'train_losses': train_losses,
    }
    
    if return_predictions:
        result.update({
            'y_true': eval_result['y_true'],
            'y_pred': eval_result['predictions'],
            'probabilities': eval_result['probabilities'],
            'logits': eval_result['logits']
        })
    
    return result

def create_dataset_without_sentiment(csv_path: str) -> str:
    """Crea versión del dataset sin features de sentimiento."""
    df = pd.read_csv(csv_path)
    
    # Mantener solo retorno_log_merval y booleano_merval
    cols_to_keep = ['retorno_log_merval', 'booleano_merval']
    df_no_sentiment = df[cols_to_keep].copy()
    
    output_path = csv_path.replace('.csv', '_sin_sentiment.csv')
    df_no_sentiment.to_csv(output_path, index=False)
    
    print(f"✅ Dataset sin sentimiento creado: {output_path}")
    print(f"   Features eliminadas: {len(df.columns) - len(cols_to_keep)}")
    print(f"   Features restantes: {len(cols_to_keep) - 1}")  # -1 por booleano_merval
    
    return output_path

print("✅ Funciones auxiliares definidas")


✅ Funciones auxiliares definidas


## 4. Cargar Datos


In [9]:
# Buscar automáticamente el último archivo procesado y generar CSV si es necesario
from src.LSTM.training_data import find_latest_processed_jsonl, combine_sentiment_and_financial

# 1. Buscar el último JSONL procesado
latest_jsonl = find_latest_processed_jsonl("data/procesada")
if latest_jsonl is None:
    print("❌ No se encontró ningún archivo JSONL en data/procesada/")
    print("   Ejecuta primero el pipeline completo:")
    print("   1. python src/procesamiento/run_etl.py")
    print("   2. python src/procesamiento/run_sentiment.py <archivo_preprocesado>")
else:
    print(f"📂 Archivo procesado encontrado: {latest_jsonl}")

# 2. Definir path del CSV
CSV_PATH = "src/LSTM/dataset_con_sentiment.csv"

# 3. Si el CSV no existe o es más viejo que el JSONL, generarlo
jsonl_path = Path(latest_jsonl) if latest_jsonl else None
csv_path = Path(CSV_PATH)

if jsonl_path and (not csv_path.exists() or csv_path.stat().st_mtime < jsonl_path.stat().st_mtime):
    print(f"\n🔄 Generando CSV desde el último archivo procesado...")
    try:
        combine_sentiment_and_financial(
            sentiment_jsonl_path=str(jsonl_path),
            output_csv_path=str(csv_path)
        )
        print(f"✅ CSV generado exitosamente")
    except Exception as e:
        print(f"❌ Error al generar CSV: {e}")
        raise
elif csv_path.exists():
    print(f"\n✅ CSV ya existe: {CSV_PATH}")
    print(f"   (Usando CSV existente. Para regenerar, elimínalo primero)")

# 4. Cargar el CSV
if csv_path.exists():
    df = pd.read_csv(CSV_PATH)
    print(f"\n✅ Dataset cargado: {len(df)} filas, {len(df.columns)} columnas")
    print(f"\nColumnas:")
    for col in df.columns:
        print(f"  - {col}")
    
    print(f"\nPrimeras filas:")
    display(df.head())
    
    print(f"\nEstadísticas básicas:")
    display(df.describe())
else:
    print(f"\n❌ No se pudo generar el CSV. Verifica que el JSONL procesado tenga datos válidos.")


❌ No se encontró ningún archivo JSONL en data/procesada/
   Ejecuta primero el pipeline completo:
   1. python src/procesamiento/run_etl.py
   2. python src/procesamiento/run_sentiment.py <archivo_preprocesado>

❌ No se pudo generar el CSV. Verifica que el JSONL procesado tenga datos válidos.


## 5. Configuración del Modelo


In [10]:
# Configuración para datasets pequeños
config = TrainingConfig(
    csv_path=CSV_PATH,
    lookback=5,  # Ventana temporal (días históricos)
    hidden_size=64,  # Neuronas en la capa LSTM
    epochs=20,  # Iteraciones de entrenamiento
    batch_size=8,  # Tamaño del batch
    learning_rate=1e-3,  # Tasa de aprendizaje
    dropout=0.1,  # Regularización
    seed=42,
    device="cpu"
)

print("📋 Configuración del modelo:")
print(f"   Lookback: {config.lookback} días")
print(f"   Hidden Size: {config.hidden_size}")
print(f"   Epochs: {config.epochs}")
print(f"   Batch Size: {config.batch_size}")
print(f"   Learning Rate: {config.learning_rate}")


📋 Configuración del modelo:
   Lookback: 5 días
   Hidden Size: 64
   Epochs: 20
   Batch Size: 8
   Learning Rate: 0.001


## 6. Entrenar Modelo CON Sentimiento


In [11]:
set_global_seed(config.seed)

print("🚀 Entrenando modelo CON features de sentimiento...")
print("="*60)

features_with, labels_with, feature_cols_with = load_dataset(CSV_PATH)
print(f"\nFeatures utilizadas ({len(feature_cols_with)}):")
for i, col in enumerate(feature_cols_with, 1):
    print(f"  {i}. {col}")

results_with = train_model_with_predictions(
    features_with,
    labels_with,
    config,
    return_predictions=True
)

print("\n✅ Entrenamiento completado")
print("\n📊 Métricas:")
for metric, value in results_with['metrics'].items():
    print(f"   {metric.capitalize()}: {value:.4f}")

# Mostrar algunas probabilidades
print("\n📈 Ejemplo de Probabilidades (primeras 10 predicciones):")
df_probs_example = pd.DataFrame({
    'y_true': results_with['y_true'][:10],
    'y_pred': results_with['y_pred'][:10],
    'probabilidad': results_with['probabilities'][:10]
})
df_probs_example['interpretacion'] = df_probs_example['probabilidad'].apply(
    lambda p: f"{'Subirá' if p >= 0.5 else 'Bajará'} ({p:.1%} confianza)"
)
display(df_probs_example)


🚀 Entrenando modelo CON features de sentimiento...


FileNotFoundError: [Errno 2] No such file or directory: 'src/LSTM/dataset_con_sentiment.csv'

## 7. Entrenar Modelo SIN Sentimiento (Baseline)


In [ ]:
# Crear dataset sin sentimiento
csv_no_sentiment = create_dataset_without_sentiment(CSV_PATH)

set_global_seed(config.seed)  # Mismo seed para comparación justa

print("🚀 Entrenando modelo SIN features de sentimiento (baseline)...")
print("="*60)

features_without, labels_without, feature_cols_without = load_dataset(csv_no_sentiment)
print(f"\nFeatures utilizadas ({len(feature_cols_without)}):")
for i, col in enumerate(feature_cols_without, 1):
    print(f"  {i}. {col}")

results_without = train_model_with_predictions(
    features_without,
    labels_without,
    config,
    return_predictions=True
)

print("\n✅ Entrenamiento completado")
print("\n📊 Métricas:")
for metric, value in results_without['metrics'].items():
    print(f"   {metric.capitalize()}: {value:.4f}")


## 8. Comparación de Modelos


In [ ]:
# Crear DataFrame de comparación
comparison_data = {
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Con Sentimiento': [
        results_with['metrics']['accuracy'],
        results_with['metrics']['precision'],
        results_with['metrics']['recall'],
        results_with['metrics']['f1']
    ],
    'Sin Sentimiento': [
        results_without['metrics']['accuracy'],
        results_without['metrics']['precision'],
        results_without['metrics']['recall'],
        results_without['metrics']['f1']
    ]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison['Mejora'] = df_comparison['Con Sentimiento'] - df_comparison['Sin Sentimiento']
df_comparison['Mejora %'] = (df_comparison['Mejora'] / df_comparison['Sin Sentimiento'] * 100).round(2)

print("📊 COMPARACIÓN DE MODELOS")
print("="*60)
display(df_comparison)

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(df_comparison))
width = 0.35

bars1 = ax.bar(x - width/2, df_comparison['Con Sentimiento'], width, 
               label='Con Sentimiento', alpha=0.8, color='#3498db')
bars2 = ax.bar(x + width/2, df_comparison['Sin Sentimiento'], width,
               label='Sin Sentimiento', alpha=0.8, color='#e74c3c')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparación de Métricas: Con vs Sin Sentimiento', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Métrica'])
ax.legend()
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Agregar valores en las barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Resumen de mejora
print("\n📈 RESUMEN DE MEJORA")
print("="*60)
for _, row in df_comparison.iterrows():
    sign = "+" if row['Mejora'] > 0 else ""
    print(f"{row['Métrica']}: {sign}{row['Mejora']:.4f} ({sign}{row['Mejora %']:.2f}%)")


In [ ]:
# Crear DataFrame con probabilidades y predicciones
df_probs_with = pd.DataFrame({
    'y_true': results_with['y_true'],
    'y_pred': results_with['y_pred'],
    'probabilidad': results_with['probabilities'],
    'correcto': results_with['y_true'] == results_with['y_pred']
})

df_probs_without = pd.DataFrame({
    'y_true': results_without['y_true'],
    'y_pred': results_without['y_pred'],
    'probabilidad': results_without['probabilities'],
    'correcto': results_without['y_true'] == results_without['y_pred']
})

print("📊 PROBABILIDADES DEL MODELO")
print("="*60)
print("\nPrimeras 20 predicciones con probabilidades (CON Sentimiento):")
display(df_probs_with.head(20))

print("\n📈 DISTRIBUCIÓN DE PROBABILIDADES")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma de probabilidades - Con sentimiento
axes[0, 0].hist(df_probs_with['probabilidad'], bins=30, alpha=0.7, color='#3498db', edgecolor='black')
axes[0, 0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Umbral (0.5)')
axes[0, 0].set_title('Distribución de Probabilidades (Con Sentimiento)', fontweight='bold')
axes[0, 0].set_xlabel('Probabilidad')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Histograma de probabilidades - Sin sentimiento
axes[0, 1].hist(df_probs_without['probabilidad'], bins=30, alpha=0.7, color='#e74c3c', edgecolor='black')
axes[0, 1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Umbral (0.5)')
axes[0, 1].set_title('Distribución de Probabilidades (Sin Sentimiento)', fontweight='bold')
axes[0, 1].set_xlabel('Probabilidad')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Box plot por clase real - Con sentimiento
df_probs_with.boxplot(column='probabilidad', by='y_true', ax=axes[1, 0])
axes[1, 0].set_title('Probabilidad por Clase Real (Con Sentimiento)', fontweight='bold')
axes[1, 0].set_xlabel('Clase Real (0=Baja, 1=Sube)')
axes[1, 0].set_ylabel('Probabilidad')
axes[1, 0].set_ylim([0, 1])

# Box plot por clase real - Sin sentimiento
df_probs_without.boxplot(column='probabilidad', by='y_true', ax=axes[1, 1])
axes[1, 1].set_title('Probabilidad por Clase Real (Sin Sentimiento)', fontweight='bold')
axes[1, 1].set_xlabel('Clase Real (0=Baja, 1=Sube)')
axes[1, 1].set_ylabel('Probabilidad')
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Estadísticas de probabilidades
print("\n📈 ESTADÍSTICAS DE PROBABILIDADES")
print("="*60)
print("\nCon Sentimiento:")
print(df_probs_with['probabilidad'].describe())
print("\nSin Sentimiento:")
print(df_probs_without['probabilidad'].describe())

# Análisis de confianza
print("\n🎯 ANÁLISIS DE CONFIANZA")
print("="*60)
for threshold in [0.6, 0.7, 0.8, 0.9]:
    high_conf_with = ((df_probs_with['probabilidad'] >= threshold) | 
                      (df_probs_with['probabilidad'] <= 1-threshold))
    if high_conf_with.sum() > 0:
        acc_high_conf = df_probs_with[high_conf_with]['correcto'].mean()
        print(f"\nCon Sentimiento - Probabilidad ≥ {threshold} o ≤ {1-threshold}:")
        print(f"  Casos: {high_conf_with.sum()} ({high_conf_with.sum()/len(df_probs_with)*100:.1f}%)")
        print(f"  Accuracy en estos casos: {acc_high_conf:.4f}")


In [ ]:
# Calcular matrices de confusión
cm_with = confusion_matrix(results_with['y_true'], results_with['y_pred'])
cm_without = confusion_matrix(results_without['y_true'], results_without['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz con sentimiento
sns.heatmap(cm_with, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Baja', 'Sube'], yticklabels=['Baja', 'Sube'])
axes[0].set_title('Con Sentimiento', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Matriz sin sentimiento
sns.heatmap(cm_without, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Baja', 'Sube'], yticklabels=['Baja', 'Sube'])
axes[1].set_title('Sin Sentimiento', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicho')

plt.tight_layout()
plt.show()

# Interpretación
print("\n📋 INTERPRETACIÓN DE MATRICES DE CONFUSIÓN")
print("="*60)
print("\nCon Sentimiento:")
print(f"  Verdaderos Negativos (TN): {cm_with[0,0]} - Predijo baja y bajó")
print(f"  Falsos Positivos (FP): {cm_with[0,1]} - Predijo sube pero bajó")
print(f"  Falsos Negativos (FN): {cm_with[1,0]} - Predijo baja pero subió")
print(f"  Verdaderos Positivos (TP): {cm_with[1,1]} - Predijo sube y subió")

print("\nSin Sentimiento:")
print(f"  Verdaderos Negativos (TN): {cm_without[0,0]}")
print(f"  Falsos Positivos (FP): {cm_without[0,1]}")
print(f"  Falsos Negativos (FN): {cm_without[1,0]}")
print(f"  Verdaderos Positivos (TP): {cm_without[1,1]}")


## 11. Curvas de Pérdida durante Entrenamiento


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_with['train_losses'], label='Con Sentimiento', linewidth=2, color='#3498db')
ax.plot(results_without['train_losses'], label='Sin Sentimiento', linewidth=2, color='#e74c3c')
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Curva de Pérdida durante Entrenamiento', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📉 ANÁLISIS DE CONVERGENCIA")
print("="*60)
print(f"\nCon Sentimiento:")
print(f"  Loss inicial: {results_with['train_losses'][0]:.4f}")
print(f"  Loss final: {results_with['train_losses'][-1]:.4f}")
print(f"  Reducción: {(1 - results_with['train_losses'][-1]/results_with['train_losses'][0])*100:.2f}%")

print(f"\nSin Sentimiento:")
print(f"  Loss inicial: {results_without['train_losses'][0]:.4f}")
print(f"  Loss final: {results_without['train_losses'][-1]:.4f}")
print(f"  Reducción: {(1 - results_without['train_losses'][-1]/results_without['train_losses'][0])*100:.2f}%")


## 12. Análisis de Hiperparámetros (Grid Search Simplificado)


In [ ]:
# Grid search simplificado (solo algunos parámetros para no tardar mucho)
param_grid = {
    'lookback': [3, 5, 7],
    'hidden_size': [32, 64],
    'learning_rate': [1e-4, 1e-3]
}

import itertools

print("🔍 Grid Search de Hiperparámetros")
print("="*60)
print(f"Total de combinaciones: {len(param_grid['lookback']) * len(param_grid['hidden_size']) * len(param_grid['learning_rate'])}")
print()

results_grid = []

for lookback, hidden_size, lr in itertools.product(
    param_grid['lookback'],
    param_grid['hidden_size'],
    param_grid['learning_rate']
):
    config_test = TrainingConfig(
        csv_path=CSV_PATH,
        lookback=lookback,
        hidden_size=hidden_size,
        epochs=10,  # Menos épocas para grid search
        batch_size=8,
        learning_rate=lr,
        dropout=0.1,
        seed=42,
        device="cpu"
    )
    
    try:
        set_global_seed(42)
        features, labels, _ = load_dataset(CSV_PATH)
        result = train_model_with_predictions(features, labels, config_test, return_predictions=False)
        
        results_grid.append({
            'lookback': lookback,
            'hidden_size': hidden_size,
            'learning_rate': lr,
            'accuracy': result['metrics']['accuracy'],
            'f1': result['metrics']['f1']
        })
        
        print(f"✅ Lookback={lookback}, Hidden={hidden_size}, LR={lr}: "
              f"Acc={result['metrics']['accuracy']:.4f}, F1={result['metrics']['f1']:.4f}")
    except Exception as e:
        print(f"❌ Error con Lookback={lookback}, Hidden={hidden_size}, LR={lr}: {e}")

df_grid = pd.DataFrame(results_grid)

if len(df_grid) > 0:
    print("\n📊 RESULTADOS DEL GRID SEARCH")
    print("="*60)
    display(df_grid.sort_values('f1', ascending=False))
    
    # Mejores parámetros
    best = df_grid.loc[df_grid['f1'].idxmax()]
    print("\n🏆 MEJORES PARÁMETROS")
    print("="*60)
    print(f"  Lookback: {best['lookback']}")
    print(f"  Hidden Size: {int(best['hidden_size'])}")
    print(f"  Learning Rate: {best['learning_rate']}")
    print(f"  Accuracy: {best['accuracy']:.4f}")
    print(f"  F1 Score: {best['f1']:.4f}")
    
    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Impacto de lookback
    grouped_lookback = df_grid.groupby('lookback')['f1'].mean()
    grouped_lookback.plot(kind='bar', ax=axes[0], color='#3498db')
    axes[0].set_title('Impacto de Lookback en F1 Score', fontweight='bold')
    axes[0].set_xlabel('Lookback')
    axes[0].set_ylabel('F1 Score Promedio')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Impacto de hidden_size
    grouped_hidden = df_grid.groupby('hidden_size')['f1'].mean()
    grouped_hidden.plot(kind='bar', ax=axes[1], color='#e74c3c')
    axes[1].set_title('Impacto de Hidden Size en F1 Score', fontweight='bold')
    axes[1].set_xlabel('Hidden Size')
    axes[1].set_ylabel('F1 Score Promedio')
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Crear tabla completa con todas las probabilidades
print("📊 TABLA COMPLETA DE PROBABILIDADES Y PREDICCIONES")
print("="*60)
print("\nModelo CON Sentimiento:")
print(f"Total de predicciones: {len(df_probs_with)}")

# Agregar interpretación
df_probs_with['interpretacion'] = df_probs_with.apply(
    lambda row: f"{'✅' if row['correcto'] else '❌'} {'Subirá' if row['probabilidad'] >= 0.5 else 'Bajará'} ({row['probabilidad']:.1%} confianza) | Real: {'Subió' if row['y_true'] == 1 else 'Bajó'}",
    axis=1
)

display(df_probs_with[['y_true', 'y_pred', 'probabilidad', 'correcto', 'interpretacion']])

# Estadísticas por rango de probabilidad
print("\n📈 ANÁLISIS POR RANGO DE PROBABILIDAD")
print("="*60)

ranges = [
    (0.0, 0.3, "Muy Baja (0-30%)"),
    (0.3, 0.5, "Baja (30-50%)"),
    (0.5, 0.7, "Alta (50-70%)"),
    (0.7, 1.0, "Muy Alta (70-100%)")
]

for min_prob, max_prob, label in ranges:
    mask = (df_probs_with['probabilidad'] >= min_prob) & (df_probs_with['probabilidad'] < max_prob)
    if mask.sum() > 0:
        subset = df_probs_with[mask]
        accuracy = subset['correcto'].mean()
        print(f"\n{label}:")
        print(f"  Casos: {mask.sum()} ({mask.sum()/len(df_probs_with)*100:.1f}%)")
        print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        print(f"  Probabilidad promedio: {subset['probabilidad'].mean():.3f}")


## 14. Interpretación de Resultados y Conclusiones


In [ ]:
print("📋 RESUMEN Y CONCLUSIONES")
print("="*60)

print("\n1. COMPARACIÓN DE MODELOS:")
print(f"   • Modelo con sentimiento tiene {len(feature_cols_with)} features")
print(f"   • Modelo sin sentimiento tiene {len(feature_cols_without)} features")
print(f"   • Mejora en Accuracy: {df_comparison.iloc[0]['Mejora']:.4f} ({df_comparison.iloc[0]['Mejora %']:.2f}%)")
print(f"   • Mejora en F1: {df_comparison.iloc[3]['Mejora']:.4f} ({df_comparison.iloc[3]['Mejora %']:.2f}%)")

print("\n2. INTERPRETACIÓN DE PROBABILIDADES:")
print("   • Probabilidad = confianza del modelo (0-1)")
print("   • Probabilidad ≥ 0.5 → Predice 'subirá'")
print("   • Probabilidad < 0.5 → Predice 'bajará'")
print(f"   • Probabilidad promedio (con sentimiento): {df_probs_with['probabilidad'].mean():.3f}")
print(f"   • Probabilidad promedio (sin sentimiento): {df_probs_without['probabilidad'].mean():.3f}")
print(f"   • Desviación estándar (con sentimiento): {df_probs_with['probabilidad'].std():.3f}")
print(f"   • Desviación estándar (sin sentimiento): {df_probs_without['probabilidad'].std():.3f}")

print("\n3. INTERPRETACIÓN DE ACCURACY:")
print(f"   • Accuracy (con sentimiento): {results_with['metrics']['accuracy']:.2%}")
print(f"   • Accuracy (sin sentimiento): {results_without['metrics']['accuracy']:.2%}")
if results_with['metrics']['accuracy'] > 0.5:
    print("   • El modelo es mejor que lanzar una moneda (50%)")
if results_with['metrics']['accuracy'] > 0.6:
    print("   • El modelo tiene buen rendimiento (>60%)")

print("\n4. DÓNDE VER LAS PROBABILIDADES:")
print("   • En la celda anterior (Sección 13) puedes ver TODAS las probabilidades")
print("   • Cada fila muestra: y_true, y_pred, probabilidad, si fue correcto")
print("   • La probabilidad indica la confianza del modelo en cada predicción")

print("\n5. RECOMENDACIONES:")
if df_comparison.iloc[0]['Mejora'] > 0:
    print("   ✅ Las features de sentimiento mejoran el modelo")
    print("   ✅ Continuar usando sentimiento en el pipeline")
else:
    print("   ⚠️  Las features de sentimiento no mejoran significativamente")
    print("   ⚠️  Considerar: más datos, mejor modelo de sentimiento, o diferentes features")

print("\n6. PRÓXIMOS PASOS:")
print("   • Recolectar más datos para mejorar el modelo")
print("   • Probar diferentes arquitecturas (GRU, Transformer)")
print("   • Fine-tuning del modelo de sentimiento")
print("   • Agregar más features (indicadores técnicos, volumen)")
print("   • Usar las probabilidades para decisiones de trading (solo si probabilidad > 0.7)")
